# Run TSM

This notebook runs TSM based on the alignment hypothesis files computed in 02_RunExperiment.ipynb

In [ ]:
from tqdm.notebook import tqdm
import numpy as np
import os
from utils.tsm import to_tsm_path

## Run TSM for different lags

In [ ]:
def run_sim_tsm(scenarios_dir, EXP_ROOT_DIR, lag):
    """
    Runs TSM on the alignment hypothesis files.

    Inputs:
    scenarios_dir: the directory containing the scenario info files
    EXP_ROOT_DIR: the root directory for the experiment
    lag: the lag to apply to the TSM
    
    Outputs:
    None, but saves the TSM path to the output directory
    """
    for scenario_id in tqdm(os.listdir(scenarios_dir)):
        hypFile = f'{EXP_ROOT_DIR}/{scenario_id}/hyp.npy'
        if not os.path.exists(hypFile):
            print(f'{hypFile} does not exist')
            continue
        hyp = np.load(hypFile)
        lag_id = f'_lag{lag}' if lag > 0 else ''
        tsm_path = to_tsm_path(hyp * 22050 / 512, lag = lag) # convert to frames
        tsm_path = tsm_path * 512 / 22050 # convert to seconds
        tsm_path[1] += hyp[1][0]
        tsm_output_path = f'{EXP_ROOT_DIR}/{scenario_id}/tsm{lag_id}.npy'
        np.save(tsm_output_path, tsm_path)

In [ ]:
systems = ['OLTW'] #['DTW','NOA', 'MATCH']
lags = [0, 100, 200, 300, 400, 500]

for system_id in range(len(systems)):
    EXP_NAME = systems[system_id]
    EXP_ROOT_DIR = f'experiments/{EXP_NAME}'
    scenarios_dir = f'scenarios'
    
    for lag in lags:
        print(f'Running {EXP_NAME} with lag {lag}...')
        run_sim_tsm(scenarios_dir, EXP_ROOT_DIR, lag)

## Generate Sonifications

In [ ]:
# from utils.tsm.generate_sonification import getModRefVQuery
# getModRefVQuery(EXP_ROOT_DIR, modes,mode_id, scenarios_dir)